### Import Dependencies

In [44]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree


In [45]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

### Embedding Function

In [46]:
@traceable(
    name='embed_query',
    run_type='embedding',
    metadata={
        'ls_provider': 'openai',
        'ls_model_name': 'text-embedding-3-small',
    }
)
def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            'input_tokens': response.usage.prompt_tokens,
            'total_tokens': response.usage.total_tokens
        }

    return response.data[0].embedding

### Retrieval Function

In [47]:
qdrant_client = QdrantClient(url='http://localhost:6333')

In [48]:
@traceable(
    name='retrieve_data',
    run_type='retriever',
)
def retrieve_data(query, collection_name='amazon-items-collection-01', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }

### Format retrieved context function

In [49]:
@traceable(
    name='process_context',
    run_type='prompt',
)
def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk, rating in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts'], retrieve_context['retrieved_context_ratings']):
        formatted_context += f"- Product ID: {id}, Product Rating: {rating}, Product Description: {chunk}\n"

    return formatted_context

### Build the prompt template

In [50]:
@traceable(
    name='build_prompt',
    run_type='prompt',
)
def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt

### Generate the answer function

In [51]:
@traceable(
    name='generate_answer',
    run_type='llm',
    metadata={
        'ls_provider': 'openai',
        'ls_model_name': 'gpt-5.4-nano',
    }
)
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort='none'
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata['usage_metadata'] = {
            'input_tokens': response.usage.prompt_tokens,
            'output_tokens': response.usage.completion_tokens,
            'total_tokens': response.usage.total_tokens
        }

    return response.choices[0].message.content

### Compile RAG pipeline

In [52]:
@traceable(
    name='rag_pipeline',
    run_type='llm',
)
def rag_pipeline(question, topk=5):
    retrieve_context = retrieve_data(query=question, k=topk)
    formatted_context = process_context(retrieve_context)

    prompt = build_prompt(question=question, formatted_context=formatted_context)
    answer = generate_answer(prompt)

    return answer

In [53]:
print(rag_pipeline('Could you suggest me a case for my Iphone 12?', 10))

For your iPhone 12, an option we have is an iPhone-compatible Apple MFi certified Lightning charger cable (not a phone case): Product ID B0BBVJJRHD (3-pack, 6ft each, supports fast charging up to 3A).

If you meant an iPhone 12 case/cover, tell me the type you want (for example: slim, rugged/protective, wallet, MagSafe) and what style/color you prefer—then I can check the available products list again.
